In [1]:
print("Hello, World!")

Hello, World!


In [2]:
%pwd

'd:\\NEW Projects\\RAG-Based-Medical-ChatBot\\Experiment'

In [3]:
import os
os.chdir('D:/NEW Projects/RAG-Based-Medical-ChatBot')

In [4]:
# from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
# def load_files(data):
#     loader = DirectoryLoader(data, glob='*.pdf',loader_cls=PyPDFLoader)
#     documents = loader.load()
#     return documents

In [6]:
# extracted_data = load_files(data='Data/')


In [7]:
# def spilt_text(extracted_data):
#     text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
#     chunks = text_splitter.split_documents(extracted_data)
#     return chunks

In [8]:
# chunks = spilt_text(extracted_data)


In [9]:
# len(chunks)

In [10]:
# chunks[0].page_content

In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [12]:
def download_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [13]:
embeddings = download_embeddings()

C:\Users\trish\AppData\Local\Temp\ipykernel_17728\2372466182.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
d:\NEW Projects\RAG-Based-Medical-ChatBot\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [14]:
results = embeddings.embed_query("hii")
len(results)

384

In [15]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [16]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
NVIDIA_NIM = os.getenv("NVIDIA_NIM")
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["NVIDIA_NIM"] = NVIDIA_NIM

In [17]:
from pinecone import Pinecone, ServerlessSpec

# pc = Pinecone()
index_name = "medical-chatbot"

# pc.create_index(
#     name=index_name,
#     dimension=384,
#     metric="cosine",
#     spec=ServerlessSpec(
#         cloud="aws",
#         region="us-east-1"
#     )
# )

In [18]:
# from langchain_pinecone import PineconeVectorStore
# search=PineconeVectorStore.from_documents(chunks, embeddings, index_name=index_name)

In [19]:
from langchain_pinecone import PineconeVectorStore
search=PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embeddings)

In [20]:
# retriever = search.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [21]:
# retrived_docs = retriever.invoke("What is diabetes?")
# retrived_docs

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="meta/llama-3.1-70b-instruct",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_NIM"],
    temperature=0.2
)

In [23]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank_documents(query, docs, top_k=3):

    pairs = [
        [query, doc.page_content]
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    scored_docs = list(zip(scores, docs))

    scored_docs.sort(
        key=lambda x: x[0],
        reverse=True
    )

    return [doc for score, doc in scored_docs[:top_k]]

In [24]:
from langchain_community.retrievers import BM25Retriever

def hybrid_retrieve(question, history):

    history_text = "\n".join([
        f"User: {chat['user']} Assistant: {chat['assistant']}"
        for chat in history[-3:]
    ])

    augmented_query = f"""
    Conversation:
    {history_text}

    Current Question:
    {question}
    """

    dense_docs = search.similarity_search(
        augmented_query,
        k=8
    )

    bm25 = BM25Retriever.from_documents(dense_docs)
    bm25.k = 5

    sparse_docs = bm25.invoke(question)
    merged_docs = dense_docs + sparse_docs

    unique_docs = []
    seen = set()

    for doc in merged_docs:

        text = doc.page_content

        if text not in seen:
            seen.add(text)
            unique_docs.append(doc)

    final_docs = rerank_documents(
        question,
        unique_docs,
        top_k=3
    )

    return final_docs

In [25]:
# from langchain.chains import create_retrieval_chain
# from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """You are a helpful RAG assistant for answering questions related to medical topics.
Use the following retrieved documents to provide accurate and concise answers to the user's questions. 
If you don't know the answer, say you don't know.

Retrieved context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}")
])

In [26]:
# from langchain_core.prompts import ChatPromptTemplate

# rewriteprompt = """
# Conversation:
# {chat_history}

# Current question:
# {input}

# Rewrite the question so it is self-contained.
# """

# rewrite_prompt = ChatPromptTemplate.from_messages([
#     ("system", rewriteprompt)
# ])

# question_chain = rewrite_prompt | llm

In [27]:
chat_history = []
MAX_HISTORY = 5

In [28]:
# result=question_chain.invoke({"input": "What is diabetes?", "chat_history": chat_history}).content
# result

In [29]:
# question_answering_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
# qa_chain = create_retrieval_chain(retriever, question_answering_chain)

In [30]:
def generate_answer(question, docs):

    context = "\n\n".join([
        doc.page_content
        for doc in docs
    ])


    chain = prompt | llm
    print(context)
    print("----")
    print(question)

    response = chain.invoke({
        "context": context,
        "question": question
    })

    return response.content

In [32]:
response = llm.invoke("Hello")
response.content

'\n\nHello! How can I help you today?'

In [ ]:
question = "What is diabetes?"
# history_text = "\n".join([
#     f"User: {chat['user']}\nAssistant: {chat['assistant']}"
#     for chat in chat_history
# ])

# response = qa_chain.invoke({
#     "input": question_chain.invoke({"input": question, "chat_history": chat_history}).content,
#     "chat_history": chat_history
# })
docs = hybrid_retrieve(question, chat_history)
response = generate_answer(question, docs)

print(response)

Diabetes is a disease in which a person cannot effectively use sugar in the blood. It is characterized by an inability to process sugars in the diet, due to a decrease in or total absence of insulin production. This occurs when the pancreas no longer produces enough insulin or when cells stop responding to the insulin that is produced, preventing glucose in the blood from being absorbed into the cells of the body.


In [ ]:
chat_history.append({
    "user": question,
    # "assistant": response["answer"]
    "assistant": response
})
chat_history = chat_history[-MAX_HISTORY:]

In [ ]:
question = "What are its symptoms?"

# response = qa_chain.invoke({
#     "input": question,
#     "chat_history": chat_history
# })
docs = hybrid_retrieve(question, chat_history)
response = generate_answer(question, docs)

print(response)

The symptoms of Diabetes mellitus include:

1. Frequent urination
2. Lethargy
3. Excessive thirst
4. Hunger
5. Sudden weight loss
6. Slow wound healing
7. Urinary tract infections
8. Gum disease
9. Blurred vision

It's worth noting that some people may not experience any noticeable symptoms, or the symptoms may be mild, making it difficult to detect the condition.


In [ ]:
chat_history

[{'user': 'What is diabetes?',
  'assistant': 'Diabetes is a disease in which a person cannot effectively use sugar in the blood. It is characterized by an inability to process sugars in the diet, due to a decrease in or total absence of insulin production. This occurs when the pancreas no longer produces enough insulin or when cells stop responding to the insulin that is produced, preventing glucose in the blood from being absorbed into the cells of the body.'}]